# 🤖 Enterprise RAG Assistant — Ingestion & Backend Hosting
### Run this notebook to index your repository into Pinecone AND host the FastAPI Backend in the cloud for free!

---
**Prerequisites before running:**
- Upload these files to the Colab session: `ingest.py`, `rag_pipeline.py`, `retrievers.py`, `utils.py`, `api.py`, `requirements-api.txt`


## ✅ Step 1 — Verify Files Are Uploaded
After uploading your files using the folder icon on the left sidebar, run this cell to confirm all files are present.

In [ ]:
import os

required_files = ['ingest.py', 'rag_pipeline.py', 'retrievers.py', 'utils.py', 'api.py', 'requirements-api.txt']
missing = [f for f in required_files if not os.path.exists(f)]

if missing:
    print(f'❌ Missing files: {missing}')
    print('Please upload them using the Files panel on the left sidebar.')
else:
    print('✅ All required files are present!')
    print('Files found:', os.listdir('.'))

## ✅ Step 2 — Set Your API Keys
**Fill in your keys below. Do NOT share this notebook publicly with keys filled in.**

In [ ]:
# ============================================================
# FILL IN YOUR API KEYS HERE
# ============================================================
PINECONE_API_KEY    = "YOUR_PINECONE_API_KEY_HERE"
PINECONE_INDEX_NAME = "codebase-rag"
PINECONE_CLOUD      = "aws"
PINECONE_REGION     = "us-east-1"
GROQ_API_KEY        = "YOUR_GROQ_API_KEY_HERE"
GROQ_MODEL          = "llama3-70b-8192"
COHERE_API_KEY      = "YOUR_COHERE_API_KEY_HERE"
GITHUB_REPO_URL     = "https://github.com/psf/requests.git"
CLONE_DIR           = "./cloned_repo"
# ============================================================

# Write to .env file
env_content = f"""PINECONE_API_KEY={PINECONE_API_KEY}
PINECONE_INDEX_NAME={PINECONE_INDEX_NAME}
PINECONE_CLOUD={PINECONE_CLOUD}
PINECONE_REGION={PINECONE_REGION}
GROQ_API_KEY={GROQ_API_KEY}
GROQ_MODEL={GROQ_MODEL}
COHERE_API_KEY={COHERE_API_KEY}
GITHUB_REPO_URL={GITHUB_REPO_URL}
CLONE_DIR={CLONE_DIR}
"""

with open('.env', 'w') as f:
    f.write(env_content)

print('✅ .env file created successfully!')

## ✅ Step 3 — Check GPU
Make sure Colab gave us a GPU. Go to **Runtime → Change runtime type → T4 GPU** if this fails.

In [ ]:
!nvidia-smi
import torch
print(f'\nPyTorch CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## ✅ Step 4 — Install Dependencies
This installs all required Python packages. Takes ~3 minutes.

In [ ]:
!pip install -r requirements-api.txt -q
print('\n✅ All packages installed!')

## ✅ Step 5 — Run Ingestion Pipeline
This will clone the repo, split files, and push vector embeddings to Pinecone.

**⚠️ Note: This step also creates the `docstore.pkl` file directly in Colab.**

In [ ]:
!python ingest.py

## 🚀 Step 6 — Start FastAPI & Expose using Ngrok
We will start the FastAPI backend and expose it using a free, secure tunnel via **Ngrok**.

### Instructions:
1. Sign up for a free account at **[dashboard.ngrok.com](https://dashboard.ngrok.com)** (you can log in with Google in 1 click).
2. Copy your **Authtoken** from the ngrok dashboard.
3. Paste your Authtoken in the `NGROK_AUTHTOKEN` variable in the cell below.
4. Run the cell below.
5. Copy the **`https://xxxx.ngrok-free.app`** URL printed in the output.
6. Paste this URL into your local frontend's `frontend-v2/.env` file as `VITE_API_URL`.

In [ ]:
# 1. Install pyngrok dependency
!pip install pyngrok -q

# 2. Configure Auth Token (PASTE YOUR NGROK AUTHTOKEN HERE)
NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN"

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# 3. Start uvicorn backend in the background
import subprocess
import time
print("Starting FastAPI backend...")
backend_process = subprocess.Popen(
    ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(5)

# 4. Open ngrok tunnel
print("Starting Ngrok tunnel...")
public_url = ngrok.connect(8000)
print(f"\n🚀 NGROK Tunnel Active!")
print(f"Copy this URL: {public_url.public_url}")